In [1]:
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig
import torch
from qwen_vl_utils import process_vision_info

W0717 22:54:09.347000 28660 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


In [2]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,  # Set False for 8-bit
    bnb_4bit_compute_dtype=torch.float16
)

In [4]:
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    "Qwen/Qwen2.5-VL-3B-Instruct", torch_dtype="auto", device_map="auto", quantization_config=bnb_config,
)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [5]:
processor = AutoProcessor.from_pretrained("Qwen/Qwen2.5-VL-3B-Instruct")

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
You have video processor config saved in `preprocessor.json` file which is deprecated. Video processor configs should be saved in their own `video_preprocessor.json` file. You can rename the file or load and save the processor back which renames it automatically. Loading from `preprocessor.json` will be removed in v5.0.


In [ ]:
import os, csv, re
image_folder = './data/train/cloth'
start_from = "05232_00.jpg"
csv_file = 'new_data.csv'
anomaly_file = 'anomaly.csv'
start_processing = False


# Loop through images
for filename in os.listdir(image_folder):
    if filename.lower().endswith(('.jpg')):
        if not start_processing:
            if filename == start_from:
                start_processing = True
                continue
            else:
                continue 
        
        path = os.path.join(image_folder, filename)
        img_name = os.path.basename(path)

        cloth_path = f"./data/train/cloth/{img_name}"
        image_path = f"./data/train/image/{img_name}"

        tagging_prompt = """Please tag the cloth in the image in terms of brand, sleeve, neckline, primary color, secondary color, and casuality.
        Examples: 1.Calvin Klein,long sleeve,v-neck,black,brown,formal 2.Levi's,short sleeve,round neck,blue,white,casual

        Note: If brand name is not present, please use "Unknown" as the brand name. Do not add new schema.
        Please use the following format: [tag1, tag2, tag3, ...].
        Do not include any other text in your response."""

        messages = [
            {
                "role": "user",
                "content": [
                    {
                        "type": "image",
                        "image": cloth_path
                    },
                    {"type": "text", "text": tagging_prompt},
                ],
            }
        ]

        text = processor.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        image_inputs, video_inputs = process_vision_info(messages)
        inputs = processor(
            text=[text],
            images=image_inputs,
            videos=video_inputs,
            padding=True,
            return_tensors="pt",
        )
        inputs = inputs.to("cuda")

        with torch.no_grad():
            generated_ids = model.generate(**inputs, max_new_tokens=128)

        generated_ids_trimmed = [
            out_ids[len(in_ids) :] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
        ]
        description_text = processor.batch_decode(
            generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
        )
        tag_text = description_text[0]

        del inputs, image_inputs, video_inputs, generated_ids, generated_ids_trimmed
        torch.cuda.empty_cache()
        ###############################

        attr_prompt = """Please tag the person's attributes in the image in terms of fit, pants color, and hair color.
        Examples: 1.loose fit,red,black 2.tight fit,black,blonde

        Please use the following format: [tag1, tag2, tag3, ...].
        Do not include any other text in your response."""

        messages = [
            {
                "role": "user",
                "content": [
                    {
                        "type": "image",
                        "image": image_path
                    },
                    {"type": "text", "text": attr_prompt},
                ],
            }
        ]

        text = processor.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        image_inputs, video_inputs = process_vision_info(messages)
        inputs = processor(
            text=[text],
            images=image_inputs,
            videos=video_inputs,
            padding=True,
            return_tensors="pt",
        )
        inputs = inputs.to("cuda")


        with torch.no_grad():
            generated_ids = model.generate(**inputs, max_new_tokens=128)

        generated_ids_trimmed = [
            out_ids[len(in_ids) :] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
        ]
        description_text = processor.batch_decode(
            generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
        )
        attribute_text = description_text[0]

        del inputs, image_inputs, video_inputs, generated_ids, generated_ids_trimmed
        torch.cuda.empty_cache()

        to_strip = "[]"
        tag_text = tag_text.strip(to_strip)
        attribute_text = attribute_text.strip(to_strip)
        data_text = img_name + "," + tag_text + "," + attribute_text
        cleaned_text = re.sub(r'[\[\]]', '', data_text)
        cleaned_text = re.sub(r',\s+', ',', cleaned_text)
        cleaned_text = cleaned_text.lower()

        row_list = cleaned_text.split(',')
        if len(row_list) == 10:
            with open(csv_file, mode='a', newline='') as f:
                writer = csv.writer(f)
                writer.writerow(row_list)
        else:
            with open(anomaly_file, mode='a', newline='') as f:
                writer = csv.writer(f)
                writer.writerow(row_list)


In [6]:
def get_new_column_value(row):
    img_name = row['img']
    image_path = f"./data/train/image/{img_name}"

    attr_prompt = """Describe the dress in a single line."""

    messages = [
        {
            "role": "user",
            "content": [
                {
                    "type": "image",
                    "image": image_path
                },
                {"type": "text", "text": attr_prompt},
            ],
        }
    ]

    text = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
    )
    inputs = inputs.to("cuda")


    with torch.no_grad():
        generated_ids = model.generate(**inputs, max_new_tokens=128)

    generated_ids_trimmed = [
        out_ids[len(in_ids) :] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
    ]
    description_text = processor.batch_decode(
        generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
    )
    description = description_text[0]
    

    del inputs, image_inputs, video_inputs, generated_ids, generated_ids_trimmed
    torch.cuda.empty_cache()
    
    return description

In [8]:
import csv, os
start_from = "01778_00.jpg"
start_processing = False

input_file = 'new_data.csv'
output_file = 'description.csv'
write_header = not os.path.exists(output_file) or os.path.getsize(output_file) == 0

with open(input_file, 'r', newline='') as infile, open(output_file, 'a', newline='') as outfile:
    reader = csv.DictReader(infile)
    fieldnames = ['img', 'description']  # Only keep these two columns
    writer = csv.DictWriter(outfile, fieldnames=fieldnames)

    if write_header:
        writer.writeheader()

    for row in reader:
        if not start_processing:
            if row['img'] == start_from:
                start_processing = True
                continue
            else:
                continue  

        new_row = {
            'img': row['img'],
            'description': get_new_column_value(row)
        }
        print(row['img'])
        writer.writerow(new_row)


01779_00.jpg
01781_00.jpg
01782_00.jpg
01783_00.jpg
01784_00.jpg
01785_00.jpg
01786_00.jpg
01787_00.jpg
01788_00.jpg
01789_00.jpg
01790_00.jpg
01791_00.jpg
01792_00.jpg
01793_00.jpg
01794_00.jpg
01795_00.jpg
01797_00.jpg
01798_00.jpg
01799_00.jpg
01800_00.jpg
01803_00.jpg
01805_00.jpg
01806_00.jpg
01807_00.jpg
01808_00.jpg
01810_00.jpg
01811_00.jpg
01813_00.jpg
01816_00.jpg
01817_00.jpg
01819_00.jpg
01821_00.jpg
01822_00.jpg
01823_00.jpg
01824_00.jpg
01825_00.jpg
01826_00.jpg
01828_00.jpg
01829_00.jpg
01830_00.jpg
01831_00.jpg
01835_00.jpg
01836_00.jpg
01837_00.jpg
01838_00.jpg
01840_00.jpg
01841_00.jpg
01842_00.jpg
01844_00.jpg
01845_00.jpg
01846_00.jpg
01847_00.jpg
01848_00.jpg
01849_00.jpg
01851_00.jpg
01852_00.jpg
01855_00.jpg
01856_00.jpg
01857_00.jpg
01860_00.jpg
01862_00.jpg
01863_00.jpg
01864_00.jpg
01865_00.jpg
01866_00.jpg
01867_00.jpg
01868_00.jpg
01869_00.jpg
01870_00.jpg
01871_00.jpg
01873_00.jpg
01876_00.jpg
01878_00.jpg
01879_00.jpg
01880_00.jpg
01882_00.jpg
01883_00.jpg

KeyboardInterrupt: 